# 1. Dataset

In [1]:
import torch
from torch.utils.data import Dataset
import numpy as np
from PIL import Image
from torch import randint
import os
import random
from torchvision import transforms
import pandas as pd



def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


mean = [0.7635  , 0.5461, 0.5705 ]
std = [0.1412 , 0.1529 , 0.1703]
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomApply(torch.nn.ModuleList([transforms.ColorJitter(), ]), p=0.3),
        transforms.RandomApply(torch.nn.ModuleList([transforms.GaussianBlur(kernel_size=3), ]), p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

danger_levels_to_id = {
    'NV': 1,       # Nevus
    'DF': 2,       # Dermatofibroma
    'BKL': 3,    # Actinic Keratosis
    'VASC': 4,     # Vascular Lesions
    'AKIEC': 5,      # Basal Cell Carcinoma (BCC)
    'BCC': 6,      # Squamous Cell Carcinoma (SCC)
    'MEL': 7       # Melanoma
}

def get_non_zero_columns(row, columns):
    return [danger_levels_to_id[col] for col in columns if row[col] != 0][0]

class ISICDataset(Dataset):
    def __init__(self,
                data_path = "/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Training_Input",
                meta_data = "/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Training_GroundTruth.csv",
                phase = "train",
                transform = None,
                seed = None):
        self.phase = phase
        self.data_path = data_path
        self.transform = data_transforms[self.phase] if (transform == None) else transform

        df = pd.read_csv(meta_data)
        columns_to_check = df.columns[1:]
        self.data = df[['image']].copy()
        self.data['label'] = df.apply(lambda row: get_non_zero_columns(row, columns_to_check), axis=1)

    def __len__(self):
        return len(self.data.index)

    def __getitem__(self, index):
        image_path = os.path.join(self.data_path, self.data['image'].iloc[index] + ".jpg")
        image = Image.open(image_path)
        image = self.transform(image)
        label = torch.tensor(self.data['label'].iloc[index] - 1, dtype=torch.long)
        return image, label

# 2. Base model

In [2]:
from torch import nn
def get_default_fc(in_features=2048, model='siamese1', ncriteria=10):
    if(model=='siamese1'):
        ret =  nn.Sequential(torch.nn.Linear(in_features, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, 64),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(64, 1))
    else:
        ret =  nn.Sequential(torch.nn.Linear(in_features, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, 64),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(64, ncriteria))
    return ret
class ResNetSimCLR(nn.Module):

    def __init__(self, base_model, out_dim):
        super(ResNetSimCLR, self).__init__()
        self.resnet_dict = {"resnet18": models.resnet18(weights='ResNet18_Weights.DEFAULT', num_classes=out_dim),
                            "resnet50": models.resnet50(weights='ResNet50_Weights.DEFAULT', num_classes=out_dim),
                            "resnet101": models.resnet101(weights='ResNet101_Weights.DEFAULT', num_classes=out_dim),
                            "densenet121": models.densenet121(weights='DenseNet121_Weights.DEFAULT', num_classes=out_dim)}

        self.backbone = self._get_basemodel(base_model)
        dim_mlp = self.backbone.fc.in_features

        # add mlp projection head
        self.backbone.fc = nn.Sequential(nn.Linear(dim_mlp, dim_mlp), nn.ReLU(), self.backbone.fc)

    def _get_basemodel(self, model_name):
        try:
            model = self.resnet_dict[model_name]
        except KeyError:
            raise InvalidBackboneError(
                "Invalid backbone architecture. Check the config file and pass one of: resnet18 or resnet50")
        else:
            return model

    def forward(self, x):
        return self.backbone(x)

In [3]:
from torchvision import models, transforms

def get_feature_extractor(feature_extractor = 'resnet50', fcnet = None, cotrain=True, ncriteria=10, model='siamese1', simclr = None):
    if(feature_extractor == 'resnet50'):    
        fextractor = models.resnet50(weights='ResNet50_Weights.DEFAULT')
        in_features = 2048
        if(simclr):
            print('load simclr resnet')
            ressimclr = ResNetSimCLR('resnet50', 1000)
            state_dict = torch.load(simclr)
            ressimclr.load_state_dict(state_dict['state_dict'])
            fextractor = ressimclr.backbone
        fextractor.fc = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
    elif(feature_extractor == 'resnet101'):    
        fextractor = models.resnet101(weights='ResNet101_Weights.DEFAULT')
        in_features = 2048
        if(simclr):
            print('load simclr resnet')
            ressimclr = ResNetSimCLR('resnet101', 1000)
            state_dict = torch.load(simclr)
            ressimclr.load_state_dict(state_dict['state_dict'])
            fextractor = ressimclr.backbone
        fextractor.fc = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
    elif(feature_extractor == 'densnet121'):
        fextractor = models.densenet121(weights='DenseNet121_Weights.DEFAULT')
        in_features = 1024
        fextractor.classifier = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
        # fextractor._modules['classifier'] = fextractor._modules.pop('classifier')
    elif(feature_extractor == 'vgg19'):
        fextractor = models.vgg19()
        fextractor.load_state_dict(torch.load('./pretrained/vgg19-dcbb9e9d.pth'))
        in_features = 25088 # https://www.geeksforgeeks.org/vgg-16-cnn-model/ length of vgg19
        fextractor.classifier = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet 
        # fextractor._modules['fc'] = fextractor._modules.pop('classifier')
    elif(feature_extractor == 'vit16'):
        fextractor = models.vit_b_16()
        in_features = 768
        fextractor.load_state_dict(torch.load('./pretrained/vit_b_16-c867db91.pth'))
        fextractor.heads.head = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet 
        # fextractor.classifier = get_default_fc(in_features) if (fcnet == None) else fcnet
    else:
        assert False, 'No feature extractor founded'

    for param in fextractor.parameters():
            param.requires_grad = cotrain
    if(feature_extractor == 'resnet50' or feature_extractor == 'resnet101'):        
        for param in fextractor.fc.parameters():
            param.requires_grad = True
    elif(feature_extractor == 'vit16'):
        for param in fextractor.heads.parameters():
            param.requires_grad = True
    else:
        for param in fextractor.classifier.parameters():
            param.requires_grad = True

    return fextractor

In [4]:
class SiameseNetwork101(nn.Module):
    """
    Siamese neural network
    Modified from: https://hackernoon.com/facial-similarity-with-siamese-networks-in-pytorch-9642aa9db2f7
    Siamese ResNet-101 from Pytorch library
    """ 
    def __init__(self):
        super(SiameseNetwork101, self).__init__()
        # note that resnet101 requires 3 input channels, will repeat grayscale image x3
        self.cnn1 = get_feature_extractor(feature_extractor='resnet50', cotrain=False)# , simclr='/mnt/c/Users/PCM/Dropbox/pretrained/SimCLR/checkpoint_10_02102023.pth.tar')
        self.cnn1.fc = nn.Sequential(torch.nn.Linear(2048, 1000),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(1000, 256))
    
    def forward_once(self, x):
        output = self.cnn1(x)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

In [5]:
class SeverityModel(nn.Module):
    """
    Siamese neural network
    Modified from: https://hackernoon.com/facial-similarity-with-siamese-networks-in-pytorch-9642aa9db2f7
    Siamese ResNet-101 from Pytorch library
    """ 
    def __init__(self, path2pretrained=''):
        super(SeverityModel, self).__init__()
        # note that resnet101 requires 3 input channels, will repeat grayscale image x3
        self.bestsimese50simclr = SiameseNetwork101()
        if (path2pretrained):
            state_dict = torch.load(path2pretrained)
            self.bestsimese50simclr.load_state_dict(state_dict["model_state_dict"])
        self.bestsimese50simclr.cnn1.add_module('fc2',
            nn.Sequential(torch.nn.Linear(256, 256),
                          torch.nn.ReLU(),
                        torch.nn.Dropout(0.1),
                        torch.nn.Linear(256, 256)))
    
    def forward_once(self, x):
        output = self.bestsimese50simclr.cnn1.fc2(self.bestsimese50simclr.cnn1(x))
        return output

    def forward(self, input1, input2, refinput):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        refinput = self.bestsimese50simclr.cnn1(refinput)
        return output1, output2, refinput

# 3. Loss function

In [6]:
from torchvision.ops.focal_loss import sigmoid_focal_loss

def Focal_loss(class_logits,  labels):
    if class_logits.numel() == 0:
        return class_logits.new_zeros([1])[0]

    N = class_logits.shape[0]
    K = class_logits.shape[1] 

    target = class_logits.new_zeros(N, K)
    target[range(len(labels)), labels] = 1
    loss = sigmoid_focal_loss(class_logits, target, reduction = 'mean')
    return loss

# 4. Pipeline

In [7]:
config = {
    "train_annotation_data_path":"/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Training_GroundTruth.csv",
    "train_image_folder_path": "/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Training_Input",
    "valid_annotation_data_path":"/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Validation_GroundTruth.csv",
    "valid_image_folder_path": "/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Validation_Input",
    "test_annotation_data_path":"/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Test_GroundTruth.csv",
    "test_image_folder_path": "/mnt/d/AiThings/SimCLRxConPro/Dataset/ISIC/ISIC2018_Task3_Test_Input",
    "batch_size":16,
    "pretrain_encoder_checkpoint": "/mnt/d/AiThings/SimCLRxConPro/upstream_task/ISIC/foundation_model/supcon-n/best.pt",
    "num_epoch": 30,
    "checkpoint": "/mnt/d/AiThings/SimCLRxConPro/output/ISIC/Supcon-n",
    "repeat": 5

}

In [8]:
image_datasets = {
    'train': ISICDataset(data_path = config["train_image_folder_path"], meta_data = config["train_annotation_data_path"], phase = "train", seed = 2),
    'val': ISICDataset(data_path = config["valid_image_folder_path"], meta_data = config["valid_annotation_data_path"], phase = "val", seed = 2),
    'test': ISICDataset(data_path = config["test_image_folder_path"], meta_data = config["test_annotation_data_path"], phase = "test", seed = 2)
}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=config["batch_size"], shuffle=True, pin_memory = True, drop_last = True)
              for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val',  'test']}
class_names = [i for i in range(1,8)]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, class_names)
print(dataset_sizes)

cuda [1, 2, 3, 4, 5, 6, 7]
{'train': 10014, 'val': 193, 'test': 1512}


In [9]:
import torch
checkpoint = torch.load(config["pretrain_encoder_checkpoint"])

basemodel = SiameseNetwork101()
basemodel.load_state_dict(checkpoint["model_state_dict"])
classifierModel = basemodel.cnn1


# basemodel = SeverityModel()
# basemodel.load_state_dict(checkpoint["model_state_dict"])
# classifierModel = basemodel.bestsimese50simclr.cnn1
# del classifierModel.fc2

classifierModel.fc = nn.Sequential(torch.nn.Linear(2048, 1000),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(1000, 256),
                                # torch.nn.Linear(256, 256),
                                # torch.nn.ReLU(),
                                # torch.nn.Dropout(0.1),
                                # torch.nn.Linear(256, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, len(class_names)))


default_cls_model = classifierModel

/tmp/ipykernel_789275/1128313052.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(config["pretrain_encoder_checkpoint"])


In [10]:
import torch.optim as optim
from torch.optim import lr_scheduler



In [11]:
from sklearn.metrics import f1_score
from tqdm import tqdm

# bestmodel = siamese50simclr
for i in range(1, config["repeat"] + 1):
    torch.cuda.empty_cache()
    momentum = 0.9
    lr = 8e-1
    optimizer_ft = optim.SGD([{'params': default_cls_model.fc.parameters()}], lr=lr, momentum=momentum)
    loss_fn= Focal_loss
    scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=10, gamma=0.5)

    for param in default_cls_model.parameters():
        param.requires_grad = False
    for param in default_cls_model.fc.parameters():
        param.requires_grad = True
    print("*"*100)
    print(f"Sample{i}")
    classifierModel = default_cls_model.to(device)
    f1max = 0
    for e in range(config["num_epoch"]):
        torch.cuda.empty_cache()
        training_acc = 0
        val_acc = 0
        training_loss_test = 0.0

        for inputs, labels in tqdm(dataloaders['train']):
            torch.cuda.empty_cache()
            classifierModel.train()
            inputs = inputs.to(device)
            labels = labels.to(device)
            # zero the parameter gradients
            optimizer_ft.zero_grad()
            outputs = classifierModel(inputs)
            _, preds = torch.max(outputs, 1)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer_ft.step()
            training_loss_test += loss.item() * inputs.size(0)
            training_acc += torch.sum(preds == labels.data)
        predlist = []
        labelist = []
        for inputs, labels in dataloaders['val']:
            torch.cuda.empty_cache()
            classifierModel.eval()
            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                outputs = classifierModel(inputs)
                _, preds = torch.max(outputs, 1)
                loss = loss_fn(outputs, labels)
            labelist.append(labels.detach().cpu().numpy()*1)
            predlist.append(preds.detach().cpu().numpy())
            val_acc += torch.sum(preds == labels.data)
        labelist = np.concatenate(labelist).ravel()
        predlist = np.concatenate(predlist).ravel()
        f1 = f1_score(predlist, labelist, average ='macro')
        if(f1 >= f1max):
            f1max = f1
            print(f"New best mode at epoch {e}")
            torch.save(classifierModel.state_dict(), os.path.join(config["checkpoint"], "best.pt"))
        torch.save(classifierModel.state_dict(), os.path.join(config["checkpoint"], "last.pt"))
        scheduler.step()


        print(f"E{e} With LR {optimizer_ft.param_groups[0]['lr']} training acc: ", training_acc.detach().cpu().numpy() / dataset_sizes['train'], "Val acc: ", val_acc.detach().cpu().numpy() / dataset_sizes['val'], "traning loss: ", training_loss_test / dataset_sizes['train'], "f1", f1)

    # %% [markdown] {"papermill":{"duration":0.192337,"end_time":"2024-08-01T04:36:20.554804","exception":false,"start_time":"2024-08-01T04:36:20.362467","status":"completed"},"tags":[]}
    # # 5. Evaluation

    # %% [code] {"papermill":{"duration":0.266946,"end_time":"2024-08-01T04:36:20.953394","exception":false,"start_time":"2024-08-01T04:36:20.686448","status":"completed"},"tags":[]}
    classifierModel.load_state_dict(torch.load(os.path.join(config["checkpoint"], "best.pt")))
    classifierModel = classifierModel.to(device)

    # %% [code] {"papermill":{"duration":141.115195,"end_time":"2024-08-01T04:38:42.201135","exception":false,"start_time":"2024-08-01T04:36:21.085940","status":"completed"},"tags":[]}
    test_acc = 0
    predlist = []
    labelist = []
    problist = []
    test_embeddings = torch.zeros((0, 2048))
    fextractor = torch.nn.Sequential(*(list(classifierModel.children())[:-1]))
    sedis = 0
    for inputs, labels in dataloaders['test']:
        classifierModel.eval()
        inputs = inputs.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            outputs = classifierModel(inputs)
            emb = fextractor(inputs)
            _, preds = torch.max(outputs, 1)
            loss = loss_fn(outputs, labels)
            sedis = sedis + torch.sum(torch.exp(torch.abs(labels - torch.max(outputs, 1)[1])))
        problist.append(outputs[:,1].detach().cpu().numpy())
        labelist.append(labels.detach().cpu().numpy()*1)
        predlist.append(preds.detach().cpu().numpy())
        # test_embeddings  = torch.cat((test_embeddings, emb.detach().cpu().flatten().unsqueeze(0)), axis=0)
        test_acc += torch.sum(preds == labels.data)

    labelist = np.concatenate(labelist).ravel()
    problist = np.concatenate(problist).ravel()
    predlist = np.concatenate(predlist).ravel()
    # test_embeddings = np.array(test_embeddings)

    # %% [code] {"papermill":{"duration":0.22059,"end_time":"2024-08-01T04:38:42.555147","exception":false,"start_time":"2024-08-01T04:38:42.334557","status":"completed"},"tags":[]}
    print(sedis/dataset_sizes['test'])

    # %% [code] {"papermill":{"duration":0.138653,"end_time":"2024-08-01T04:38:42.824209","exception":false,"start_time":"2024-08-01T04:38:42.685556","status":"completed"},"tags":[]}
    print("test_acc acc: ", test_acc / dataset_sizes['test'])


    # %% [code] {"papermill":{"duration":0.153887,"end_time":"2024-08-01T04:38:43.108296","exception":false,"start_time":"2024-08-01T04:38:42.954409","status":"completed"},"tags":[]}
    from sklearn.metrics import classification_report
    from sklearn.metrics import roc_auc_score

    print(classification_report(labelist, predlist, digits=3))

****************************************************************************************************
Sample1


100%|██████████| 625/625 [03:03<00:00,  3.41it/s]


New best mode at epoch 0
E0 With LR 0.8 training acc:  0.6808468144597564 Val acc:  0.6994818652849741 traning loss:  0.022272090520393546 f1 0.2144776495662836


100%|██████████| 625/625 [03:02<00:00,  3.42it/s]


New best mode at epoch 1
E1 With LR 0.8 training acc:  0.7100059916117436 Val acc:  0.7409326424870466 traning loss:  0.019141670306785534 f1 0.2965862822274952


...

E28 With LR 0.2 training acc:  0.9353904533652886 Val acc:  0.8652849740932642 traning loss:  0.005124847408006868 f1 0.6399544764125535


100%|██████████| 625/625 [03:00<00:00,  3.47it/s]


E29 With LR 0.1 training acc:  0.9339924106251248 Val acc:  0.8652849740932642 traning loss:  0.005124935451238404 f1 0.634242805638854


/tmp/ipykernel_789275/486507950.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  classifierModel.load_state_dict(torch.load(os.path.join(config["checkpoint"], "best.pt")

tensor(29.3512, device='cuda:0')
test_acc acc:  tensor(0.7566, device='cuda:0')
              precision    recall  f1-score   support

           0      0.874     0.894     0.884       903
           1      0.500     0.341     0.405        44
           2      0.658     0.667     0.662       216
           3      0.464     0.371     0.413        35
           4      0.446     0.674     0.537        43
           5      0.558     0.462     0.506        93
           6      0.574     0.547     0.560       170

    accuracy                          0.761      1504
   macro avg      0.582     0.565     0.567      1504
weighted avg      0.757     0.761     0.757      1504



In [12]:
# !pip install matplotlib
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# cm = confusion_matrix(labelist, predlist)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
# disp.plot()
# plt.savefig("/kaggle/working/confusion_matrix.png")
# plt.show()